# Create new Salesforce Job__c + Worksite Location (Account)

This trigger creates a **new worksite Account** (location) and then creates a **new Job__c** that points at it.

## Modes
- **Manual mode (recommended):** set `USE_SUPABASE = False` and edit `MANUAL_JOB_ROW`.
- **Supabase mode:** set `USE_SUPABASE = True` and set `SUPABASE_JOB_ID`.

## Safety / test mode
- Set `PROXI_SF_TEST_MODE=true` in `.env` to automatically make `External_Job_ID__c` and `Job_Client_Job_Id__c` unique for repeated testing.
- Set `DRY_RUN = True` to preview without creating anything.

## Notes
- This notebook **does not require Supabase** for the worksite creation step (it creates the Account directly in Salesforce and uses the returned Id in the Job create payload).
- `PROXI_SF_CREATE_WORKSITES=true` must be set.
- Writes must be enabled (see `utils.sf_write_flags.proxi_sf_writes_enabled`).


In [12]:
import os, sys, json
from pathlib import Path

project_root = Path.cwd().resolve()
for _ in range(15):
    if (project_root / "src" / "utils").is_dir():
        break
    project_root = project_root.parent
else:
    raise RuntimeError("Could not locate repo root containing src/utils")

sys.path.insert(0, str(project_root / "src"))
from dotenv import load_dotenv
load_dotenv(project_root / ".env")
print("Ready")


Ready


In [13]:
# --- Change these ---
# Option A: Create using a real Supabase job_current row
USE_SUPABASE = False
SUPABASE_JOB_ID = "19596"
SUPABASE_SCHEMA = "public"

# Option B: Create a brand-new Job__c + a brand-new worksite without Supabase
# - Set job_id to "AUTO" to generate a unique id each run (recommended).
# - Keep practice_value in the "2174 - Farmington, MO" format.
# - City/state are also used to key the worksite location map.
MANUAL_JOB_ROW = {
    "job_id": "AUTO",
    "city": "Farmington",
    "state": "MO",
    "address_line": "101 W Main St, Farmington, MO",
    "practice_value": "2174 - Farmington, MO",
    "status": "Closed",  # always Closed for this trigger
    "insight": "",
    "dates_needed": "",
    "standard_schedule": "",
    "types_of_cases": "",
    "support_staff": "",
    "avg_patients_per_day": "",
    "roster_only": "false",
    "job_ranking": "B",
    "description_full_text": "",
    "point_of_contact": "",
}

DRY_RUN = False


In [14]:
from datetime import datetime

from utils.supabase_db import load_job_current_row_for_salesforce


def _auto_job_id() -> str:
    # Keep it short so External_Job_ID__c stays within org limits.
    return datetime.utcnow().strftime("%y%m%d%H%M%S")


if USE_SUPABASE:
    job_row = load_job_current_row_for_salesforce(SUPABASE_JOB_ID, schema=SUPABASE_SCHEMA)
else:
    job_row = dict(MANUAL_JOB_ROW)
    jid = str(job_row.get("job_id") or "").strip()
    if jid.upper() == "AUTO" or not jid:
        job_row["job_id"] = _auto_job_id()

# Always force Closed for this trigger.
job_row["status"] = "Closed"

print(
    f"Using job_id={job_row.get('job_id')}  city={job_row.get('city')}  state={job_row.get('state')}  "
    f"address_line={job_row.get('address_line')}  practice_value={job_row.get('practice_value')}  status={job_row.get('status')}"
)


Using job_id=260417092714  city=Farmington  state=MO  address_line=101 W Main St, Farmington, MO  practice_value=2174 - Farmington, MO  status=Closed


In [ ]:
from utils.salesforce import get_token_auto

token = get_token_auto(
    os.environ["SALESFORCE_CONSUMER_KEY"],
    os.environ["SALESFORCE_CONSUMER_SECRET"],
    os.environ.get("SALESFORCE_USERNAME") or None,
    os.environ.get("SALESFORCE_PASSWORD") or None,
    use_client_credentials=os.environ.get("SALESFORCE_USE_USERNAME_PASSWORD", "").lower() not in ("1", "true", "yes"),
    security_token=os.environ.get("SALESFORCE_SECURITY_TOKEN") or None,
    use_sandbox=os.environ.get("SALESFORCE_USE_SANDBOX", "").lower() in ("1", "true", "yes"),
    token_url=os.environ.get("SALESFORCE_TOKEN_URL") or "https://proxi.my.salesforce.com",
)
INSTANCE_URL = token["instance_url"]
ACCESS_TOKEN = token["access_token"]
print("Authenticated:", INSTANCE_URL)


In [ ]:
from utils.address_display_format import (
    format_us_address_line_for_display,
    strip_redundant_city_state_from_shipping_street,
)
from utils.sf_job_rest_minimal import create_account_record, describe_sobject, filter_createable_fields
from utils.sf_push_defaults import (
    SF_ACCOUNT_ASPEN_DENTAL_MANAGEMENT_ID,
    format_worksite_account_name,
    worksite_account_record_type_id,
)
from utils.sf_write_flags import proxi_sf_writes_enabled


def _env_truthy(name: str) -> bool:
    return os.environ.get(name, '').strip().lower() in ('1', 'true', 'yes')


if not proxi_sf_writes_enabled():
    raise RuntimeError('Salesforce writes are disabled (check utils.sf_write_flags / env).')
if not _env_truthy('PROXI_SF_CREATE_WORKSITES'):
    raise RuntimeError('Set PROXI_SF_CREATE_WORKSITES=true to enable Account creation.')

city = (job_row.get('city') or '').strip()
state = (job_row.get('state') or '').strip()
if not city or not state:
    raise ValueError('MANUAL_JOB_ROW must include non-empty city/state to create worksite Account.')

ship = (job_row.get('address_line') or '').strip()
ship_street = format_us_address_line_for_display(ship) if ship else None
if ship_street == '':
    ship_street = None
if ship_street:
    dedup = strip_redundant_city_state_from_shipping_street(ship_street, city=city, state=state)
    if dedup:
        ship_street = dedup
if ship_street == '':
    ship_street = None

body = {
    'Name': format_worksite_account_name(city, state),
    'ParentId': SF_ACCOUNT_ASPEN_DENTAL_MANAGEMENT_ID,
    # These drive formula fields that display "Worksite 1 Address" on Job__c.
    'ShippingCity': city,
    'ShippingState': state,
}
if ship_street:
    body['ShippingStreet'] = ship_street

acct_desc = describe_sobject(INSTANCE_URL, ACCESS_TOKEN, 'Account')
rt_id = worksite_account_record_type_id(acct_desc)
if rt_id:
    body['RecordTypeId'] = rt_id

fields = filter_createable_fields(acct_desc, body)
if not fields.get('Name'):
    fields['Name'] = body['Name']

print('Account create payload:')
print(json.dumps(fields, indent=2, default=str))

if DRY_RUN:
    print('\nDRY_RUN — no Account create. Set DRY_RUN=False in config cell and rerun.')
    worksite_id = None
else:
    resp = create_account_record(INSTANCE_URL, ACCESS_TOKEN, fields)
    worksite_id = (resp.get('id') or '').strip() or None
    if not worksite_id:
        raise RuntimeError(f'Account create returned no id: {resp!r}')
    print('\nCreated Account →', worksite_id)

if worksite_id:
    job_row['sf_worksite_account_id'] = worksite_id


Account create payload:
{
  "Name": "Aspen Dental - Farmington, MO",
  "ParentId": "0015f00000HH63kAAD",
  "ShippingStreet": "101 W Main St",
  "RecordTypeId": "0125f000000ZbkhAAC"
}

Created Account → 001UP00000e2JcEYAU


In [ ]:
from utils.sf_job_payload import prepare_payload_for_write
from utils.sf_job_rest_minimal import create_job_record, describe_sobject

JOB_OBJECT = os.environ.get("SALESFORCE_JOB_OBJECT", "Job__c").strip()
describe = describe_sobject(INSTANCE_URL, ACCESS_TOKEN, JOB_OBJECT)

fields = prepare_payload_for_write(
    job_row,
    describe,
    use_canonical_description=True,
    for_update=False,
    description_use_html=True,
)

# Optional test-only mutations
if os.environ.get("PROXI_SF_TEST_MODE", "").lower() in ("1", "true", "yes"):
    import uuid
    suffix = uuid.uuid4().hex[:6].upper()
    if "External_Job_ID__c" in fields:
        base = str(fields.get("External_Job_ID__c") or "").strip()
        base_digits = "".join(ch for ch in base if ch.isdigit())[-10:]
        fields["External_Job_ID__c"] = f"T{suffix}{base_digits}"  # <= 17 chars
    if "Job_Client_Job_Id__c" in fields and fields.get("Job_Client_Job_Id__c"):
        fields["Job_Client_Job_Id__c"] = f"{fields['Job_Client_Job_Id__c']} (T{suffix})"
    if "Job_Client_Job_Description__c" in fields:
        fields["Job_Client_Job_Description__c"] = "[TEST RECORD] " + (fields["Job_Client_Job_Description__c"] or "")

show = dict(fields)
dk = "Job_Client_Job_Description__c"
if dk in show and len(str(show[dk])) > 500:
    show[dk] = str(show[dk])[:500] + f"... ({len(str(fields[dk]))} chars)"
print(json.dumps(show, indent=2, default=str))

if DRY_RUN:
    print("\nDRY_RUN — set DRY_RUN = False in cell 2 and re-run from there.")
else:
    result = create_job_record(INSTANCE_URL, ACCESS_TOKEN, JOB_OBJECT, fields)
    new_id = result.get("id", "(unknown)")
    print(f"\nCreated {JOB_OBJECT} → {new_id}")


Skipped (not createable on object): Occupation_DJC__c
{
  "External_Job_ID__c": "T4492900417091911",
  "Job_Account__c": "0015f00000HH63kAAD",
  "Job_Worksite_Location_1__c": "001UP00000e2JcEYAU",
  "Job_Client_Job_Id__c": "2174 - Farmington, MO (T449290)",
  "Job_Client_Job_Description__c": "[TEST RECORD] <p><strong>General Dentist Locum Tenens Opportunity in Farmington, MO</strong></p><p><br/></p><p>We are seeking a General Dentist for a locum tenens opportunity in Farmington, Missouri. This position offers the opportunity to practice comprehensive general dentistry with a supportive clinical team and steady patient flow.</p><p><br/></p><p>Travel and lodging may be available for qualified candidates.</p><p><strong>Pay Range:</strong> Starting at $125/hour</p><p><br/></p><p><strong>... (883 chars)",
  "External_Job_Link__c": "https://portal.kimedics.com/app/workspace/job-posts/260417091911",
  "Job_Status__c": "Open",
  "Job_State__c": "Missouri",
  "Job_City__c": "Farmington",
  "Sal